## Wearable Style Dataset Analysis

In [118]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Initialisation 
df = pd.read_csv("wearable_data.csv")
# Notebook display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Plot styling
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

# # Initial data inspection
# print("\nData shape is:", df.shape)
# print("\nData types are:", df.dtypes)
# print("\nAmount of null:", df.isna().sum())
# print("\nAmount of duplicated:", df.duplicated().sum())
# print("\n Desribe data:", df.describe())

# # Function to describe categorical data
# def profile_categoricals(df, max_unique=20):
#     # Print value counts for categorical columns. Skip high-cardinality columns
#     cat_cols = df.select_dtypes(include=["object", "category"]).columns

#     for col in cat_cols:
#         n_unique = df[col].nunique(dropna=False)
#         print(f"\n{'=' * 40}")
#         print(f"  {col}  ({n_unique} unique values)")
#         print(f"{'=' * 40}")

#         if n_unique > max_unique:
#             print(f"  [Skipped - too many unique values to be useful]")
#         else:
#             print(df[col].value_counts(dropna=False))

# profile_categoricals(df)
# print(df.head(10))


## Remove Duplicate Rows
The dataset contained 16 exact duplicate rows. These reperesent artifacts of the the wearable's data collection. Since the rows are exact copies, we can safely keep the first occurrence of each and drop the rest. This is a lossless operation.

As all duplicates are exact copies, drop_duplicates() with default settings can be used, no need for subsets.

In [119]:
# Data cleaning

# Show all duplicated rows
def check_duplicates(data_frame):
    dupes = data_frame[data_frame.duplicated(keep=False)].sort_values(["user_id", "timestamp"])
    print("Total duplicated rows:", len(dupes))
    display(dupes.head(10))
    return dupes
check_duplicates(df)
df = df.drop_duplicates().reset_index(drop=True)
check_duplicates(df);

Total duplicated rows: 32


,user_id,date,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent
59,U001,2024-01-12,2024-01-12 12:10:00,NaN,76.90,8.62,Good,Yoga,NaN,40,98.80
157,U001,2024-01-12,2024-01-12 12:10:00,NaN,76.90,8.62,Good,Yoga,NaN,40,98.80
15,U001,2024-01-30,2024-01-30 18:55:00,3643.00,75.00,6.77,Good,NaN,163.40,31,97.10
259,U001,2024-01-30,2024-01-30 18:55:00,3643.00,75.00,6.77,Good,NaN,163.40,31,97.10
71,U001,2024-02-28,2024-02-28 11:56:00,1163.00,320.00,7.90,Excellent,NaN,53.30,15,100.20
295,U001,2024-02-28,2024-02-28 11:56:00,1163.00,320.00,7.90,Excellent,NaN,53.30,15,100.20
247,U002,2024-01-14,2024-01-14 15:35:00,6963.00,79.60,7.49,Excellent,NaN,258.50,55,NaN
275,U002,2024-01-14,2024-01-14 15:35:00,6963.00,79.60,7.49,Excellent,NaN,258.50,55,NaN
26,U002,2024-01-19,2024-01-19 08:07:00,7997.00,65.40,6.69,Poor,NaN,339.60,78,99.80
193,U002,2024-01-19,2024-01-19 08:07:00,7997.00,65.40,6.69,Poor,NaN,339.60,78,99.80


Total duplicated rows: 0


,user_id,date,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent


## Date-time formatting

Due to the multiple date formats present in "date" and "timestamp", pandas converted both columns to an "object" datatype. The "timestamp" column contains both time and date information, making the "date" column redundant. 

Before dropping the "date" column, I converted it to datetime format, and checked it against the dates in "timestamp". This will reveal if the data is actually duplicated. If so, the column will be dropped, because it's fully derivable from timestamp and duplicated storage invites drift.  

In [120]:
# Check NaN quantity before date-time parsing
# print("Amount of null in timestamp column:", df["timestamp"].isna().sum())
# print("Amount of null in date column:", df["date"].isna().sum())

# Parsing
def date_time_parsing:
    return NaN
df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", dayfirst=True)
df["date"] = pd.to_datetime(df["date"], format="mixed", dayfirst=True)

# Checking date-time parsing
# print("\nAmount of null in timestamp column:", df["timestamp"].isna().sum())
# print("Amount of null in date column:", df["date"].isna().sum())
# display(df["timestamp"].head(10))
# display(df["date"].head(10))
# print("Min timestamp:", df["timestamp"].min())
# print("Max timestamp:", df["timestamp"].max())
# print("\nValue counts by hour of day (top 5):")
# print(df["timestamp"].dt.hour.value_counts().head())

# Verify that the date part of timestamp matches the date column
dates_match = (df["timestamp"].dt.normalize() == df["date"]).all()
print(f"Date components match: {dates_match}")

if dates_match:
    df = df.drop(columns="date")
    print("Dropped redundant 'date' column")
else:
    mismatches = (df["timestamp"].dt.normalize() == df["date"])
    print("Number of mismatches =", (len(df) - mismatches.sum()))
    print(df[["timestamp", "date"]][df["timestamp"].dt.normalize() != df["date"]])

# Check results
df.head(10)

Date components match: False
Number of mismatches = 20
              timestamp       date
16  2024-02-04 20:14:00 2024-04-02
18  2024-01-06 18:37:00 2024-06-01
24  2024-01-04 14:39:00 2024-04-01
53  2024-01-08 18:48:00 2024-08-01
89  2024-01-05 11:25:00 2024-05-01
103 2024-02-09 18:48:00 2024-09-02
104 2024-01-12 08:51:00 2024-12-01
125 2024-01-09 10:07:00 2024-09-01
130 2024-02-11 19:57:00 2024-11-02
139 2024-02-10 15:34:00 2024-10-02
155 2024-02-12 21:59:00 2024-12-02
157 2024-02-09 19:38:00 2024-09-02
203 2024-01-07 06:40:00 2024-07-01
211 2024-01-10 16:28:00 2024-10-01
217 2024-02-08 08:41:00 2024-08-02
226 2024-02-11 18:40:00 2024-11-02
248 2024-01-02 20:39:00 2024-02-01
277 2024-02-01 17:40:00 2024-01-02
284 2024-02-08 19:34:00 2024-08-02
296 2024-02-12 13:14:00 2024-12-02


,user_id,date,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent
0,U003,2024-02-16,2024-02-16 21:31:00,12855.00,82.90,6.94,Good,Running,500.00,101,97.60
1,U004,2024-11-01,2024-11-01 08:53:00,7121.00,0.00,6.26,Good,NaN,320.50,55,97.90
2,U001,2024-08-01,2024-08-01 19:36:00,4770.00,83.30,7.41,Fair,NaN,138.40,47,97.10
3,U002,2024-06-01,2024-06-01 11:21:00,12956.00,67.20,6.56,Excellent,Yoga,500.10,104,95.70
4,U002,2024-08-02,2024-08-02 13:07:00,14454.00,62.70,5.98,Good,Cycling,585.50,127,96.90
5,U004,2024-01-18,2024-01-18 10:36:00,6027.00,75.20,5.43,Good,Yoga,264.50,48,98.00
6,U005,2024-02-29,2024-02-29 10:29:00,11535.00,73.50,NaN,Good,strength,499.10,99,98.20
7,U003,2024-09-01,2024-09-01 12:43:00,7128.00,84.50,7.20,Fair,Cycling,258.20,60,98.50
8,U003,2024-10-02,2024-10-02 15:29:00,8680.00,78.30,6.33,Good,Yoga,337.90,73,97.30
9,U002,2024-01-25,2024-01-25 10:36:00,9608.00,63.50,NaN,Good,Cycling,364.20,76,96.10
